# Stage 2 Colab A100 executor
Upload `/content/stage2_source.zip` and a pre-recorded `/content/stage2_source_expectations.json` containing exact `archive_sha256` and `commit`.

In [ ]:
import hashlib, json, shutil, subprocess, zipfile
from pathlib import Path, PurePosixPath
source_zip=Path('/content/stage2_source.zip'); expectations=Path('/content/stage2_source_expectations.json')
expected=json.loads(expectations.read_text(encoding='utf-8'))
if set(expected)!={'archive_sha256','commit'} or len(expected['archive_sha256'])!=64 or len(expected['commit'])!=40: raise RuntimeError('invalid external source expectations')
if hashlib.sha256(source_zip.read_bytes()).hexdigest()!=expected['archive_sha256']: raise RuntimeError('outer source ZIP checksum mismatch')
unpack=Path('/content/stage2_source_unpack'); repo=Path('/content/stage2_repo'); verify=Path('/content/stage2_bundle_verify')
for path in (unpack,repo,verify):
    if path.exists(): shutil.rmtree(path)
with zipfile.ZipFile(source_zip) as z:
    names=z.namelist()
    if set(names)!={'stage2_source.bundle','source_metadata.json'} or len(names)!=len(set(names)) or any(PurePosixPath(n).is_absolute() or '..' in PurePosixPath(n).parts or '\\' in n for n in names): raise RuntimeError('unsafe source archive')
    unpack.mkdir(); [(unpack/n).write_bytes(z.read(n)) for n in names]
meta=json.loads((unpack/'source_metadata.json').read_text(encoding='utf-8')); bundle=unpack/'stage2_source.bundle'
if meta['git']['commit']!=expected['commit'] or hashlib.sha256(bundle.read_bytes()).hexdigest()!=meta['bundle_sha256']: raise RuntimeError('source metadata mismatch')
subprocess.run(['git','init',str(verify)],check=True); subprocess.run(['git','-C',str(verify),'bundle','verify',str(bundle)],check=True)
subprocess.run(['git','clone',str(bundle),str(repo)],check=True); subprocess.run(['git','-C',str(repo),'checkout','--detach',expected['commit']],check=True)
if subprocess.check_output(['git','-C',str(repo),'status','--porcelain'],text=True).strip(): raise RuntimeError('cloned source is dirty')
gpu=subprocess.check_output(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],text=True).strip().splitlines()[0]
if 'A100' not in gpu: raise RuntimeError(f'Stage 2 requires A100, found {gpu}')

In [ ]:
subprocess.run(['python','-m','pip','install','--upgrade','pip'],check=True,cwd=repo)
subprocess.run(['python','-m','pip','install','torch==2.11.0','--index-url','https://download.pytorch.org/whl/cu128'],check=True,cwd=repo)
subprocess.run(['python','-m','pip','install','-r','requirements.txt'],check=True,cwd=repo)
subprocess.run(['python','-m','unittest','discover','-s','tests','-v'],check=True,cwd=repo)
primary='ties_results/stage2_smoke/colab_a100_run1'; repeat='ties_results/stage2_smoke/colab_a100_repeat_full_sr'; canonical='ties_results/canonical_v1'
subprocess.run(['python','monitor_stage2_job.py','--events','ties_results/.stage2_monitor/colab_a100_run1.events.jsonl','--watch',primary,'--','python','run_stage2_smoke.py','--mode','primary','--environment','colab_a100','--protocol','docs/paper_rebuild/FROZEN_EXPERIMENT_PROTOCOL.md','--output-dir',primary,'--fresh'],check=True,cwd=repo)
subprocess.run(['python','validate_stage2_smoke.py','--root',primary,'--conditions','standard_lora','full_sr','class_prior_reweight','--canonical-dir',canonical],check=True,cwd=repo)
subprocess.run(['python','monitor_stage2_job.py','--events','ties_results/.stage2_monitor/colab_a100_repeat_full_sr.events.jsonl','--watch',repeat,'--','python','run_stage2_smoke.py','--mode','repeat_full_sr','--environment','colab_a100','--protocol','docs/paper_rebuild/FROZEN_EXPERIMENT_PROTOCOL.md','--output-dir',repeat,'--fresh'],check=True,cwd=repo)
subprocess.run(['python','validate_stage2_smoke.py','--root',primary,'--conditions','standard_lora','full_sr','class_prior_reweight','--canonical-dir',canonical,'--compare-repeat',repeat],check=True,cwd=repo)
subprocess.run(['python','freeze_stage2_environment.py','--protocol','docs/paper_rebuild/FROZEN_EXPERIMENT_PROTOCOL.md','--smoke-root',primary,'--repeat-root',repeat,'--source-archive','/content/stage2_source.zip','--commands',primary+'/commands.json','--repeat-commands',repeat+'/commands.json','--repo-root','.','--output-dir','ties_results/stage2_smoke/freeze_bundle','--fresh'],check=True,cwd=repo)
subprocess.run(['python','validate_stage2_smoke.py','--root',primary,'--conditions','standard_lora','full_sr','class_prior_reweight','--canonical-dir',canonical,'--compare-repeat',repeat],check=True,cwd=repo)
subprocess.run(['python','freeze_stage2_environment.py','--verify-only','--output-dir','ties_results/stage2_smoke/freeze_bundle'],check=True,cwd=repo)

In [ ]:
from canonical.freeze import build_evidence_archive
build_evidence_archive(repo, Path('/content/stage2_a100_evidence.zip'), expectations_path=expectations)